In [2]:
import pandas as pd
import numpy as np
file_path = 'Downloads/Crime_Data_from_2020_to_Present.csv'

df = pd.read_csv(file_path)
print(f"Dataset loaded with {df.shape[0]} rows.")
display(df.head(2))

# Data Preprocessing & Constraints
# Converting date strings to datetime objects
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'])
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'])

# Identifier function: Create a unique ID if one didn't exist (using index as a base)
def generate_custom_id(index):
    return f"CRIME_EVENT_{index}"

df['Internal_ID'] = df.index.map(generate_custom_id)

# Handling missing values (Constraint: Victim Age must be valid)
df = df.dropna(subset=['Vict Age'])
df = df[df['Vict Age'] > 0]  # Mathematical operation for filtering

print("Preprocessing complete.")

# String Operations
# Standardizing Area Names and extracting crime categories
df['AREA NAME'] = df['AREA NAME'].str.upper().str.strip()
df['Crime_Category'] = df['Crm Cd Desc'].str.split(' ').str[0]

display(df[['AREA NAME', 'Crime_Category']].head())

# Statistical functions, Quantile and Percentile
mean_age = df['Vict Age'].mean()
median_age = df['Vict Age'].median()
quantile_75 = df['Vict Age'].quantile(0.75)
percentile_90 = np.percentile(df['Vict Age'], 90)

print(f"Mean Age: {mean_age:.2f}")
print(f"75th Percentile: {quantile_75}")
print(f"90th Percentile: {percentile_90}")

# Aggregate functions
crime_summary = df.groupby('AREA NAME').agg({
    'Vict Age': ['mean', 'min', 'max'],
    'DR_NO': 'count'
}).rename(columns={'count': 'Total_Crimes'})

display(crime_summary.head())

# Automated dataset operation (Dynamic Trigger)
def automated_priority_trigger(dataframe):
    """
    Automatically flags areas based on crime volume.
    Areas with crime counts above the 75th percentile are 'HIGH' priority.
    """
    counts = dataframe['AREA NAME'].value_counts()
    threshold = counts.quantile(0.75)
    high_priority_areas = counts[counts >= threshold].index.tolist()
    print(f"Trigger identified {len(high_priority_areas)} high-priority areas based on data distribution.")
    dataframe['Priority'] = dataframe['AREA NAME'].apply(
        lambda x: 'HIGH' if x in high_priority_areas else 'STANDARD'
    )
    return dataframe

# Run the automated trigger
df = automated_priority_trigger(df)

# Display the results of the automated flagging
display(df[['AREA NAME', 'Priority']].drop_duplicates())

# Export operation
df.to_csv('processed_crime_data_with_priority.csv', index=False)
print("Exported processed data to processed_crime_data_with_priority.csv")

# User Input: Check Priority for a Specific Area
def check_area_priority(dataframe):
    user_input = input("Enter Area Name to check priority: ").upper().strip()
    result = dataframe[dataframe['AREA NAME'] == user_input]['Priority'].unique()
    if len(result) > 0:
        print(f"The area '{user_input}' is classified as: {result[0]} priority.")
    else:
        print(f"Area '{user_input}' not found in the dataset. Please check the spelling.")

check_area_priority(df)

Dataset loaded with 1005198 rows.


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,190326475,3/1/2020 0:00,3/1/2020 0:00,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,...,AA,Adult Arrest,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506
1,200106753,2/9/2020 0:00,2/8/2020 0:00,1800,1,Central,182,1,330,BURGLARY FROM VEHICLE,...,IC,Invest Cont,330.0,998.0,NaN,NaN,1000 S FLOWER ST,NaN,34.0444,-118.2628


Preprocessing complete.


,AREA NAME,Crime_Category
1,CENTRAL,BURGLARY
2,SOUTHWEST,BIKE
3,VAN NUYS,SHOPLIFTING-GRAND
11,HOLLENBECK,BURGLARY
19,HOLLYWOOD,PIMPING


Mean Age: 39.50
75th Percentile: 50.0
90th Percentile: 62.0


...

Trigger identified 6 high-priority areas based on data distribution.


,AREA NAME,Priority
1,CENTRAL,HIGH
2,SOUTHWEST,HIGH
3,VAN NUYS,STANDARD
11,HOLLENBECK,STANDARD
19,HOLLYWOOD,HIGH
23,WEST VALLEY,STANDARD
43,NORTHEAST,STANDARD
46,77TH STREET,HIGH
54,WILSHIRE,STANDARD
55,RAMPART,STANDARD


Exported processed data to processed_crime_data_with_priority.csv


Enter Area Name to check priority:  Foothill


The area 'FOOTHILL' is classified as: STANDARD priority.


In [3]:
from sklearn.preprocessing import LabelEncoder

le_area = LabelEncoder()
le_crime = LabelEncoder()

df['AREA_ENC'] = le_area.fit_transform(df['AREA NAME'])
df['CRIME_ENC'] = le_crime.fit_transform(df['Crime_Category'])
df['Priority_Label'] = df['Priority'].map({
    'STANDARD': 0,
    'HIGH': 1
})
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df[['Vict Age', 'AREA_ENC', 'CRIME_ENC']]
y = df['Priority_Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 1.0


In [4]:
y = df['Crime_Category']
X = df[['Vict Age', 'AREA_ENC']]
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
from sklearn.cluster import KMeans

crime_counts = (
    df.groupby('AREA NAME')
      .size()
      .reset_index(name='Crime_Count')
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42
)

crime_counts['Cluster'] = kmeans.fit_predict(
    crime_counts[['Crime_Count']]
)

display(crime_counts)

C:\Users\ashis\anaconda3_new\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,AREA NAME,Crime_Count,Cluster
0,77TH STREET,46217,1
1,CENTRAL,52099,1
2,DEVONSHIRE,29189,2
3,FOOTHILL,24561,2
4,HARBOR,27738,2
5,HOLLENBECK,24122,2
6,HOLLYWOOD,39293,0
7,MISSION,30150,2
8,N HOLLYWOOD,36159,0
9,NEWTON,33398,0


In [7]:
monthly_crimes = (
    df.groupby(['AREA NAME', pd.Grouper(key='DATE OCC', freq='ME')])
      .size()
      .reset_index(name='Crime_Count')
)
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [8]:
from sklearn.ensemble import IsolationForest

crime_counts = (
    df.groupby('AREA NAME')
      .size()
      .reset_index(name='Crime_Count')
)

iso = IsolationForest(
    contamination=0.1,
    random_state=42
)

crime_counts['Anomaly'] = iso.fit_predict(
    crime_counts[['Crime_Count']]
)